# S5Mars semantic segmentation dataset analysis

Self-contained Kaggle notebook for loading `Mirali33/mb-s5mars`, smoke-testing a PyTorch DataLoader, computing basic statistics, and saving visualizations.

In [ ]:
!pip install -q datasets huggingface_hub hydra-core omegaconf

In [ ]:
HF_TOKEN = "HF_TOKEN_PLACEHOLDER"
REPO_ID = "Mirali33/mb-s5mars"
SPLIT = "train"
IMAGE_SIZE = (512, 512)
IGNORE_INDEX = 0
MAX_SAMPLES_ANALYSIS = 50
OUTPUT_DIR = "/kaggle/working/s5mars_analysis"

In [ ]:
from huggingface_hub import HfApi, get_token, login, whoami
from huggingface_hub.utils import HfHubHTTPError

def is_hf_logged_in():
    global HF_TOKEN

    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        HF_TOKEN = user_secrets.get_secret("hf_token")
        if HF_TOKEN:
            print("Detected Kaggle environment. HF Token loaded from Kaggle Secrets.")
    except ImportError:
        print("Kaggle environment not detected. No HF Token loaded automatically.")
    except Exception as exc:
        print(f"Could not load HF token from Kaggle Secrets: {exc}")

    if HF_TOKEN in (None, "", "HF_TOKEN_PLACEHOLDER"):
        HF_TOKEN = get_token()

    if HF_TOKEN in (None, "", "HF_TOKEN_PLACEHOLDER"):
        print("No HF token available. Continuing with unauthenticated access.")
        return False

    try:
        login(token=HF_TOKEN)
        user = whoami(token=HF_TOKEN)
        username = user.get("name", "unknown user")
        print(f"HF Login successful: {username}")
        return True
    except HfHubHTTPError:
        print("HF Login failed with the provided token. Continuing unauthenticated.")
        HF_TOKEN = None
        return False

HF_LOGGED_IN = is_hf_logged_in()


In [ ]:
import json
import logging
import os
import random
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("kaggle_s5mars")
logger.info("Outputs saved to %s", OUTPUT_DIR)

In [ ]:
CLASS_NAMES = {
    0: "Background",
    1: "Bedrock",
    2: "Hole",
    3: "Ridge",
    4: "Rock",
    5: "Rover",
    6: "Sand / Soil",
    7: "Sky",
    8: "Track",
}
PALETTE = {
    0: (30, 30, 30),
    1: (166, 118, 29),
    2: (0, 109, 119),
    3: (131, 56, 236),
    4: (230, 57, 70),
    5: (255, 183, 3),
    6: (42, 157, 143),
    7: (69, 123, 157),
    8: (244, 162, 97),
}
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

In [ ]:
class SegmentationTransform:
    def __init__(self, size, normalize=True):
        self.size = size
        self.normalize = normalize

    def __call__(self, image, mask):
        image = image.resize((self.size[1], self.size[0]), Image.Resampling.BILINEAR)
        mask = mask.resize((self.size[1], self.size[0]), Image.Resampling.NEAREST)
        image_tensor = torch.from_numpy(np.asarray(image, dtype=np.float32) / 255.0).permute(2, 0, 1).contiguous()
        if self.normalize:
            image_tensor = (image_tensor - MEAN) / STD
        mask_tensor = torch.from_numpy(np.asarray(mask, dtype=np.int64)).long()
        return image_tensor.float(), mask_tensor

def denormalize(image_tensor):
    return (image_tensor.cpu() * STD + MEAN).clamp(0, 1).permute(1, 2, 0).numpy()

def colorize_mask(mask):
    mask = mask.cpu().numpy() if isinstance(mask, torch.Tensor) else np.asarray(mask)
    color = np.zeros((*mask.shape, 3), dtype=np.uint8)
    for class_id, rgb in PALETTE.items():
        color[mask == class_id] = rgb
    return color

def overlay(image, color_mask, alpha=0.45):
    base = (np.clip(image, 0, 1) * 255).astype(np.uint8)
    out = ((1 - alpha) * base + alpha * color_mask).astype(np.uint8)
    background = np.all(color_mask == PALETTE[IGNORE_INDEX], axis=-1)
    out[background] = base[background]
    return out

def add_class_legend(fig):
    handles = [
        Patch(
            facecolor=tuple(channel / 255 for channel in PALETTE[class_id]),
            edgecolor="black",
            label=f"{class_id}: {CLASS_NAMES[class_id]}",
        )
        for class_id in sorted(CLASS_NAMES)
    ]
    fig.legend(handles=handles, loc="center right", title="Classes", frameon=True)

In [ ]:
class S5MarsHFDataset(Dataset):
    def __init__(self, repo_id, split, token=None, transform=None):
        token = None if token in (None, "", "HF_TOKEN_PLACEHOLDER") else token
        logger.info("Loading %s split=%s", repo_id, split)
        self.dataset = load_dataset(repo_id, split=split, token=token)
        self.transform = transform
        required = {"image", "mask", "width", "height", "class_labels"}
        missing = required - set(self.dataset.column_names)
        if missing:
            raise ValueError(f"Missing required columns: {sorted(missing)}")
        logger.info("Loaded dataset: repo=%s split=%s samples=%d columns=%s", repo_id, split, len(self.dataset), self.dataset.column_names)

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, index):
        sample = self.dataset[index]
        image = sample["image"].convert("RGB")
        mask = sample["mask"].convert("L")
        image_tensor, mask_tensor = self.transform(image, mask)
        return {
            "image": image_tensor,
            "mask": mask_tensor,
            "class_labels": list(sample["class_labels"]),
            "width": int(sample["width"]),
            "height": int(sample["height"]),
            "index": index,
        }

In [ ]:
def segmentation_collate_fn(batch):
    return {
        "image": torch.stack([sample["image"] for sample in batch]),
        "mask": torch.stack([sample["mask"] for sample in batch]),
        "class_labels": [sample["class_labels"] for sample in batch],
        "width": torch.tensor([sample["width"] for sample in batch], dtype=torch.long),
        "height": torch.tensor([sample["height"] for sample in batch], dtype=torch.long),
        "index": torch.tensor([sample["index"] for sample in batch], dtype=torch.long),
    }

dataset = S5MarsHFDataset(REPO_ID, SPLIT, HF_TOKEN, transform=SegmentationTransform(IMAGE_SIZE))
dataloader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    collate_fn=segmentation_collate_fn,
)
logger.info("Dataset ready: %d samples", len(dataset))
logger.info("DataLoader ready: %d batches, batch_size=%d", len(dataloader), dataloader.batch_size)
batch = next(iter(dataloader))
print("image", batch["image"].dtype, batch["image"].shape)
print("mask", batch["mask"].dtype, batch["mask"].shape)
print("unique ids", sorted(batch["mask"][0].unique().tolist()))
print("class_labels", dataset[0]["class_labels"])

In [ ]:
n = min(len(dataset), MAX_SAMPLES_ANALYSIS)
image_class_counts = {class_id: 0 for class_id in CLASS_NAMES}
pixel_counts = np.zeros(len(CLASS_NAMES), dtype=np.int64)
total_pixels = 0
ignored_pixels = 0

for i in tqdm(range(n), desc="analysis"):
    sample = dataset[i]
    mask = sample["mask"].numpy()
    for class_id in np.unique(mask):
        image_class_counts[int(class_id)] = image_class_counts.get(int(class_id), 0) + 1
    pixel_counts += np.bincount(mask.reshape(-1), minlength=len(CLASS_NAMES))[: len(CLASS_NAMES)]
    total_pixels += mask.size
    ignored_pixels += int((mask == IGNORE_INDEX).sum())

image_level = pd.DataFrame([
    {"class_id": k, "class_name": CLASS_NAMES.get(k, str(k)), "num_images": v, "percentage_images": v / n}
    for k, v in sorted(image_class_counts.items())
])
valid_pixels = total_pixels - ignored_pixels
pixel_distribution = pd.DataFrame([
    {
        "class_id": k,
        "class_name": CLASS_NAMES[k],
        "pixel_count": int(v),
        "is_ignore_index": k == IGNORE_INDEX,
        "percentage_total_pixels": float(v / total_pixels) if total_pixels else 0.0,
        "percentage_valid_pixels": 0.0 if k == IGNORE_INDEX or not valid_pixels else float(v / valid_pixels),
    }
    for k, v in enumerate(pixel_counts)
])
ignore_ratio = {
    "ignore_index": IGNORE_INDEX,
    "num_samples": n,
    "ignored_pixels": ignored_pixels,
    "total_pixels": total_pixels,
    "ignore_pixel_ratio": ignored_pixels / total_pixels if total_pixels else 0.0,
}
display(image_level, pixel_distribution, ignore_ratio)

In [ ]:
def plot_one(index, axis_row=None):
    sample = dataset[index]
    image = denormalize(sample["image"])
    color_mask = colorize_mask(sample["mask"])
    over = overlay(image, color_mask)
    axes = axis_row
    if axes is None:
        fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    for ax, arr, title in zip(axes, [image, color_mask, over], ["Image", "Mask", "Overlay"]):
        ax.imshow(arr)
        ax.set_title(title)
        ax.axis("off")

indices = list(range(min(3, len(dataset))))
fig, axes = plt.subplots(len(indices), 3, figsize=(14, 4 * len(indices)))
if len(indices) == 1:
    axes = np.expand_dims(axes, 0)
for row, index in enumerate(indices):
    plot_one(index, axes[row])
add_class_legend(fig)
fig.tight_layout(rect=(0, 0, 0.84, 1))
grid_path = Path(OUTPUT_DIR) / "sample_grid.png"
fig.savefig(grid_path, dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
image_level.to_csv(Path(OUTPUT_DIR) / "image_level_class_distribution.csv", index=False)
pixel_distribution.to_csv(Path(OUTPUT_DIR) / "mask_pixel_distribution.csv", index=False)
with open(Path(OUTPUT_DIR) / "dataset_summary.json", "w") as f:
    json.dump({"repo_id": REPO_ID, "split": SPLIT, "num_samples": len(dataset), "columns": dataset.dataset.column_names}, f, indent=2)
with open(Path(OUTPUT_DIR) / "ignore_pixel_ratio.json", "w") as f:
    json.dump(ignore_ratio, f, indent=2)
print(f"Outputs saved to: {OUTPUT_DIR}")